In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

In [2]:
df = pd.read_csv("bike_trip_data(July 2025-June 2026).csv")

In [3]:
df_cleaned = df

In [4]:
df_cleaned['start_station_name'] = df_cleaned['start_station_name'].str.lower()
df_cleaned['end_station_name'] = df_cleaned['end_station_name'].str.lower()
df_cleaned = df_cleaned.drop_duplicates()

In [5]:
condition = df_cleaned['started_at'] > df_cleaned['ended_at']
df_cleaned.loc[condition, ['started_at', 'ended_at']] = df_cleaned.loc[condition, ['ended_at', 'started_at']].values

In [6]:
condition = df_cleaned[['end_station_name', 'end_lat']].isna().all(axis=1)
df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [7]:
df_cleaned['start_station_name'] = df_cleaned['start_station_name'].str.replace('[archived]', '', regex=False)
df_cleaned['end_station_name'] = df_cleaned['end_station_name'].str.replace('[archived]', '',regex=False)

In [8]:
df_cleaned = df_cleaned.drop(columns=['start_station_id', 'end_station_id'])

In [9]:
for col in df_cleaned.columns[4:6]:     # Access the 'start_station_name' and 'end_station_name' columns
    df_cleaned[col] = df_cleaned[col].str.strip()
    df_cleaned[col] = df_cleaned[col].str.replace(r'^\s+|\s+$|\s{2,}', ' ', regex=True)

In [10]:
df_cleaned

,ride_id,rideable_type,started_at,ended_at,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,member_casual
0,455D43BD91D73437,classic_bike,2025-07-05 17:15:08.456,2025-07-05 17:25:47.079,lincoln ave & diversey pkwy,lincoln ave & addison st,41.932225,-87.658617,41.946176,-87.673308,member
1,9D4A6B723ECD98CA,classic_bike,2025-07-01 13:57:38.878,2025-07-01 14:06:35.780,cottage grove ave & oakwood blvd,cottage grove ave & 47th st,41.822985,-87.607100,41.809855,-87.606755,member
2,C57044CF523302ED,classic_bike,2025-07-31 16:49:28.142,2025-07-31 17:15:28.999,theater on the lake,winthrop ave & lawrence ave,41.926277,-87.630834,41.968812,-87.657659,member
3,AFD35552E6685B6E,electric_bike,2025-07-17 09:36:21.058,2025-07-17 09:46:54.706,pine grove ave & waveland ave,winthrop ave & lawrence ave,41.949473,-87.646453,41.968812,-87.657659,member
4,C0582EBAA6CED519,classic_bike,2025-07-02 18:43:45.213,2025-07-02 18:57:06.687,theater on the lake,sheffield ave & wellington ave,41.926277,-87.630834,41.936253,-87.652662,member
...,...,...,...,...,...,...,...,...,...,...,...
5932344,86932262232F29F0,electric_bike,2026-06-17 21:09:10.026,2026-06-17 21:15:09.708,clark st & lincoln ave,NaN,41.915689,-87.634600,41.930000,-87.640000,member
5932345,6654F4B0FC4D361E,classic_bike,2026-06-07 08:07:09.330,2026-06-07 08:09:11.320,california ave & cortez st,california ave & division st,41.900363,-87.696704,41.903029,-87.697474,member
5932346,6A71125A045366A8,electric_bike,2026-06-02 15:36:13.487,2026-06-02 15:47:10.902,monticello ave & diversey ave,NaN,41.931720,-87.718460,41.950000,-87.710000,member
5932347,152C05EEB9316CD6,electric_bike,2026-06-10 15:55:03.821,2026-06-10 16:03:57.409,leavitt st & addison st,NaN,41.946655,-87.683359,41.950000,-87.710000,member


---

In [11]:
for columns in df_cleaned.columns[4:6]:
    station_ref = df_cleaned[df_cleaned[columns].notna()]
    if columns == "start_station_name":
        start_station_ref = station_ref
        start_station_ref = start_station_ref.drop_duplicates(subset=["start_lat", "start_lng"])[["start_lat", "start_lng", "start_station_name"]]
    else:
        end_station_ref = station_ref
        end_station_ref = end_station_ref.drop_duplicates(subset=["end_lat", "end_lng"])[["end_lat", "end_lng", "end_station_name"]]
        
    if columns == "start_station_name":
        start_station_ref = start_station_ref.drop_duplicates(subset=[columns])
        start_station_ref = start_station_ref.reset_index()
        start_station_ref = start_station_ref.drop(columns=['index'])
    else:
        end_station_ref_station_ref = end_station_ref.drop_duplicates(subset=[columns])
        end_station_ref = end_station_ref.reset_index()
        end_station_ref = end_station_ref.drop(columns=['index'])

In [12]:
# Combine start and end station references
station_ref = pd.concat([
    start_station_ref.rename(columns={'start_lat': 'lat', 'start_lng': 'lng', 'start_station_name': 'station_name'}),
    end_station_ref.rename(columns={'end_lat': 'lat', 'end_lng': 'lng', 'end_station_name': 'station_name'})
], ignore_index=True)

# Make it unique by dropping duplicates based on coordinates and station name
station_ref = station_ref.drop_duplicates(subset=['station_name']).reset_index(drop=True)

In [13]:
for columns in df_cleaned.columns[4:6]:
    mask_null = df_cleaned[columns].isna()
    df_null = df_cleaned[mask_null].copy()
    latlon_ref = np.radians(station_ref[["lat", "lng"]].values)
    if columns == 'start_station_name':
        latlon_null = np.radians(df_null[["start_lat", "start_lng"]].values)
    else:
        latlon_null = np.radians(df_null[["end_lat", "end_lng"]].values)
    tree = BallTree(latlon_ref, metric="haversine")

    distances, indices = tree.query(latlon_null, k=2)

    distances_km = distances * 6371.0
    max_distances_km = 1.8
    margin_km = 0.005

    station_result = []

    # Evaluate the nearest station prediction results
    for i in range(len(df_null)):
        dist1, dist2 = distances_km[i][0], distances_km[i][1]
        idx1, idx2 = indices[i][0], indices[i][1]

    # If the nearest station is more than 1.8 km away, leave it blank (NaN)
        if dist1 > max_distances_km:
            station_result.append(np.nan)

    # If the distance difference between candidate 1 and 2 is very small, mark as ambiguous
        elif (dist2 - dist1) <= margin_km:
            name1 = station_ref.iloc[idx1]["station_name"]
            name2 = station_ref.iloc[idx2]["station_name"]
            station_result.append(f"Ambiguous: {name1} / {name2}")

    # If safe, fill with the name of the first nearest station
        else:
            station_result.append(station_ref.iloc[idx1]["station_name"])

    # Insert the prediction results back into the main dataframe
    df_cleaned.loc[mask_null, columns] = station_result

    condition = df_cleaned[columns].isna()
    df_cleaned = df_cleaned.drop(df_cleaned[condition].index)

In [14]:
df_cleaned.rename(columns={'ride_id': 'trip_id', 'rideable_type': 'bike_type'}, inplace=True)

In [15]:
df_cleaned = df_cleaned.astype({
    'trip_id': 'str',
    'bike_type': 'category',
    'start_station_name': 'category',
    'end_station_name': 'category',
        'member_casual': 'category'
    })

In [16]:
df_cleaned['started_at'] = pd.to_datetime(df_cleaned['started_at'])
df_cleaned['ended_at'] = pd.to_datetime(df_cleaned['ended_at'])

In [17]:
df_cleaned['ride_length_min'] = (df_cleaned['ended_at'] - df_cleaned['started_at']).dt.total_seconds() / 60
df_cleaned['ride_length_min'] = df_cleaned['ride_length_min'].round(2)

df_cleaned['day_of_week'] = df_cleaned['started_at'].dt.day_name()

df_cleaned['month'] = df_cleaned['started_at'].dt.strftime('%Y-%m')

df_cleaned['start_hour'] = df_cleaned['started_at'].dt.hour

In [18]:
df_cleaned = df_cleaned.drop(columns=['start_lat', 'start_lng', 'end_lat', 'end_lng'])

In [19]:
check = df_cleaned['ride_length_min']>=2
df_cleaned = df_cleaned[check]

q1 = df_cleaned['ride_length_min'].quantile(0.25)
q3 = df_cleaned['ride_length_min'].quantile(0.75)
iqr = q3 - q1

outlier_line = q3 + (1.5 * iqr)

df_cleaned = df_cleaned[df_cleaned['ride_length_min']<= outlier_line]

In [20]:
df_cleaned.to_csv("bike_trip_data_clean(July 2025-June 2026).csv", index=False)

In [21]:
df_cleaned

,trip_id,bike_type,started_at,ended_at,start_station_name,end_station_name,member_casual,ride_length_min,day_of_week,month,start_hour
0,455D43BD91D73437,classic_bike,2025-07-05 17:15:08.456,2025-07-05 17:25:47.079,lincoln ave & diversey pkwy,lincoln ave & addison st,member,10.64,Saturday,2025-07,17
1,9D4A6B723ECD98CA,classic_bike,2025-07-01 13:57:38.878,2025-07-01 14:06:35.780,cottage grove ave & oakwood blvd,cottage grove ave & 47th st,member,8.95,Tuesday,2025-07,13
2,C57044CF523302ED,classic_bike,2025-07-31 16:49:28.142,2025-07-31 17:15:28.999,theater on the lake,winthrop ave & lawrence ave,member,26.01,Thursday,2025-07,16
3,AFD35552E6685B6E,electric_bike,2025-07-17 09:36:21.058,2025-07-17 09:46:54.706,pine grove ave & waveland ave,winthrop ave & lawrence ave,member,10.56,Thursday,2025-07,9
4,C0582EBAA6CED519,classic_bike,2025-07-02 18:43:45.213,2025-07-02 18:57:06.687,theater on the lake,sheffield ave & wellington ave,member,13.36,Wednesday,2025-07,18
...,...,...,...,...,...,...,...,...,...,...,...
5932344,86932262232F29F0,electric_bike,2026-06-17 21:09:10.026,2026-06-17 21:15:09.708,clark st & lincoln ave,stockton dr & wrightwood ave,member,5.99,Wednesday,2026-06,21
5932345,6654F4B0FC4D361E,classic_bike,2026-06-07 08:07:09.330,2026-06-07 08:09:11.320,california ave & cortez st,california ave & division st,member,2.03,Sunday,2026-06,8
5932346,6A71125A045366A8,electric_bike,2026-06-02 15:36:13.487,2026-06-02 15:47:10.902,monticello ave & diversey ave,troy st & grace st,member,10.96,Tuesday,2026-06,15
5932347,152C05EEB9316CD6,electric_bike,2026-06-10 15:55:03.821,2026-06-10 16:03:57.409,leavitt st & addison st,troy st & grace st,member,8.89,Wednesday,2026-06,15
